# Module 0: Technology Readiness Check

This notebook checks that your computer is ready for the course labs. It is **not graded**, and there is nothing to submit for points.

| Part | What happens | Time |
|---|---|---|
| 1 Setup checks | Python, packages, repository folders, data files and stored embeddings | about 15 min |
| 2 pandas warm-up | Short pandas steps on the course data, plus three optional tries | about 20 min |
| 3 Save and commit | Save a readiness report, then commit your work in GitHub Desktop | about 10 min |

**How to read the results.** Each check prints one line:

- `PASS`: this part is ready.
- `FIX`: something needs your attention. The next line, *What to do*, tells you how to fix it.
- `NOTE`: information only. You can continue.

## Before you start

1. Follow the setup steps on the **Technology and Accessibility** page in Canvas. You check (or install) Anaconda, GitHub Desktop and VS Code, accept the email invitation to your repository (the instructor creates it for you), clone it, and create the `ban6403` environment. You may use GitHub Codespaces in your browser instead (see `.devcontainer/README.md`).
2. Open the repository folder.
   - **Local (Anaconda):** in VS Code, choose **File > Open Folder** and open the whole repository folder, not only this file.
   - **Codespaces:** the repository folder is already open in your browser.
3. Open `notebooks/M00_Technology_Readiness.ipynb` and click the kernel picker at the top right.
   - **Local (Anaconda):** choose **ban6403**.
   - **Codespaces:** choose **Python Environments**, then the only Python 3.13 entry.
4. Run the cells in order from the top: click a cell and press **Shift+Enter**. Or choose **Run All**.

## Part 1 Setup checks

### 1.1 Start the record

This cell imports a few standard Python tools and defines `record`. Every check below calls `record`. It prints one result line and keeps it for the report you save in Part 3. Expect no output.

In [ ]:
from pathlib import Path
from datetime import datetime
from importlib.metadata import PackageNotFoundError, version
import hashlib
import json
import os
import platform
import re
import sys

# One entry per check, keyed by the item name. Part 3 saves this as a table.
checks = {}

def record(item, status, detail, fix=""):
    """Take one check result (item, PASS/FIX/NOTE, detail, fix advice); print it and keep it."""
    # A passed check needs no advice.
    if status == "PASS":
        fix = ""
    checks[item] = {"item": item, "status": status, "detail": detail, "what_to_do": fix}
    # ljust(4) pads the status to four characters, so the | lines up.
    print(status.ljust(4), "|", item + ":", detail)
    if fix:
        print("       What to do:", fix)

### 1.2 Python version and kernel

The **kernel** is the Python program that runs this notebook. The course uses Python 3.13 in an Anaconda environment named `ban6403` (or in GitHub Codespaces). The cell prints the end of the Python path, so you can see which environment runs. The full path is not printed, because it can contain your user name.

Expect `PASS` for the Python version. For the kernel, expect `PASS`:

- **Local (Anaconda):** the line shows `environment ban6403`.
- **Codespaces:** the line shows `GitHub Codespaces`.

A wrong kernel gives `FIX`. Fix it before you go on, because the later labs need the course packages.

In [ ]:
# GitHub Codespaces sets this environment variable.
in_codespaces = os.environ.get("CODESPACES") == "true"

# sys.executable is the path of the running Python; .parts[-3:] keeps its last three parts.
python_path = Path(sys.executable)
print("Python:", platform.python_version(), "| running from:", Path(*python_path.parts[-3:]))

# sys.version_info compares as numbers: (3, 13) means Python 3.13.
if sys.version_info >= (3, 13):
    python_status = "PASS"
else:
    python_status = "FIX"
record("Python version", python_status, platform.python_version() + " (need 3.13 or newer)",
       "Select the ban6403 kernel. If it is missing, create the environment (Technology and Accessibility page).")

# sys.prefix is the folder of the running environment.
if in_codespaces:
    kernel_ok, kernel_detail = True, "GitHub Codespaces"
else:
    kernel_ok, kernel_detail = "ban6403" in sys.prefix, "environment " + Path(sys.prefix).name
record("notebook kernel", "PASS" if kernel_ok else "FIX", kernel_detail,
       "This is not the ban6403 environment. Click the kernel picker at the top right, choose ban6403, then run from the top.")

### 1.3 Required packages

The required work in every lab uses five packages: numpy, pandas, scikit-learn, scipy and matplotlib. The cell looks up the installed version of each one. It compares that version with the oldest one the course supports. Newer versions are fine.

Expect five `PASS` lines. If you see `FIX`, fix it before you go on, because the later cells need these packages.

In [ ]:
MINIMUM = {"numpy": "1.24", "pandas": "3.0", "scikit-learn": "1.8", "scipy": "1.10", "matplotlib": "3.7"}
FIX_PACKAGES = ("Check that the kernel is ban6403 (step 1.2). Then, in Anaconda Prompt (Windows) or Terminal (macOS), "
                "go to the repository folder and run: conda env update -n ban6403 -f environment.yml --prune")

def major_minor(number):
    """Take a version text such as '3.0.5' and return (3, 0), so two versions compare as numbers."""
    # re.findall finds every run of digits: '3.0.5' gives ['3', '0', '5'].
    parts = re.findall(r"\d+", number)
    return (int(parts[0]), int(parts[1]))

# Keep the versions for the summary file in Part 3.
versions = {"python": platform.python_version()}
for package, minimum in MINIMUM.items():
    # version() raises PackageNotFoundError when a package is missing; that is a FIX, not a crash.
    try:
        installed = version(package)
    except PackageNotFoundError:
        record("package " + package, "FIX", "not installed", FIX_PACKAGES)
        continue
    versions[package] = installed
    ok = major_minor(installed) >= major_minor(minimum)
    record("package " + package, "PASS" if ok else "FIX",
           f"version {installed} (need {minimum} or newer)", FIX_PACKAGES)

### 1.4 Repository folder

Jupyter, VS Code and Codespaces run each notebook from its own folder, here `notebooks/`. The repository folder is one level up. So every lab opens data with a **relative path** such as `../data/course/cfpb_complaints.csv`, where `..` means "one folder up". The same code then works on every computer.

The cell checks the main folders and checks that the folder is a Git repository, so you can commit your work. It also warns if the folder sits inside a cloud-synced folder such as OneDrive.

In [ ]:
# Path.cwd() is the folder this notebook runs in (notebooks/); .parent is one level up.
REPO = Path.cwd().parent
print("Repository folder:", REPO.name, "| inside:", REPO.parent.name)

for folder in ["notebooks", "data/raw", "data/course", "data/partitions"]:
    # The / operator joins the repository folder and a subfolder into one path.
    found = (REPO / folder).is_dir()
    record("folder " + folder, "PASS" if found else "FIX", "found" if found else "not found",
           "In VS Code choose File > Open Folder and open the whole repository folder, then open this notebook from notebooks/.")

# Git keeps its history in a hidden folder called .git.
has_git = (REPO / ".git").exists()
record("Git repository", "PASS" if has_git else "FIX",
       "found" if has_git else "this folder has no Git history",
       "Clone your copy of the repository with GitHub Desktop (File > Clone Repository). A downloaded ZIP file cannot commit your work.")

# Look for the name of a cloud-sync service in the folder path.
cloud = []
for word in ["OneDrive", "Dropbox", "Google Drive", "iCloud", "Mobile Documents"]:
    if word in str(REPO):
        cloud.append(word)
# An empty list counts as False, so no cloud folder gives PASS.
record("folder location", "NOTE" if cloud else "PASS",
       "inside " + cloud[0] if cloud else "not inside a cloud-synced folder",
       "Cloud sync can lock or duplicate files while Git works. If you see odd errors, clone the repository again into Documents/GitHub.")

### 1.5 Data files: fingerprints

Everyone in the course must use exactly the same data, or results will not match. A **SHA-256 hash** is a 64-character fingerprint computed from a file's bytes. If one character in the file changes, the hash changes. The helper `text_sha256` treats Windows and Mac line endings the same way. So the same file gives the same hash on every computer.

The cell compares four files with their expected fingerprints. They are the 24 customer messages, the Module 2 partition, the 10,000 course complaints and the 22,634 clothing reviews. Expect four `PASS` lines.

The cell also sets `ready`. It is `True` when every package and data file check passed. Steps 1.6 and 1.7 read the data only when `ready` is `True`. Otherwise they record `SKIP`, and step 1.8 still shows every check with its advice. If numpy or pandas is not installed, step 1.6 stops with an error; fix the package first.

In [ ]:
EXPECTED_SHA256 = {
    "data/raw/customer_feedback_m01.csv": "de9a95b7aec38238388de50139cfa2419d0b5549ce9ba39b90bdee060156ab40",
    "data/partitions/m02_partition.csv": "2a08b6a84ba16ddc1c2225ab64936f03164c4192fa959aa7ab05b861c389189c",
    "data/course/cfpb_complaints.csv": "f6a7bfdcae91598b1e218bdb08d018da357b32715e45ba90d87bc9ff738080d7",
    "data/course/clothing_reviews.csv": "8a61265cd4b1421c95949affbc20acb39bbbbb44748daca78b11a076262e8682",
}
FIX_DATA = ("Do not edit files in data/. In GitHub Desktop, open the Changes tab. If the file is listed, right-click it "
            "and choose Discard Changes. If it is not listed, download the file again from the course template on GitHub "
            "and put it back in the same folder.")

def text_sha256(path):
    """Take a file path and return its SHA-256 fingerprint, counting Windows line endings as Mac ones."""
    # read_bytes() reads the raw file; b"\r\n" is a Windows line ending.
    data = Path(path).read_bytes().replace(b"\r\n", b"\n")
    return hashlib.sha256(data).hexdigest()

for name, expected in EXPECTED_SHA256.items():
    path = REPO / name
    if path.exists():
        same = text_sha256(path) == expected
        # expected[:12] shows the first 12 characters of the fingerprint.
        if same:
            detail = "sha256 " + expected[:12] + " matches"
        else:
            detail = "sha256 " + expected[:12] + " does NOT match"
    else:
        same, detail = False, "not found"
    record("file " + path.name, "PASS" if same else "FIX", detail, FIX_DATA)

# The data steps below need every package and every data file.
ready = True
for item, result in checks.items():
    # startswith() checks how the item name begins.
    if item.startswith(("package ", "file ")) and result["status"] == "FIX":
        ready = False
if not ready:
    print("Fix the items above, then choose Restart and Run All. Steps 1.6 and 1.7 are skipped until then.")

### 1.6 Data files: read them with pandas

A matching hash shows the bytes are right. Now read the two main files with pandas and count the rows. The course complaints come with a fixed split into `train` (6,000), `valid` (2,000) and `test` (2,000). You never change it. Expect three `PASS` lines.

In [ ]:
import numpy as np
import pandas as pd

SKIPPED = "skipped: fix the FIX items in steps 1.3 and 1.5 first"

if ready:
    feedback = pd.read_csv("../data/raw/customer_feedback_m01.csv")
    complaints = pd.read_csv("../data/course/cfpb_complaints.csv")

    # len() of a table is its number of rows.
    record("customer messages", "PASS" if len(feedback) == 24 else "FIX",
           f"{len(feedback)} rows (expected 24)", FIX_DATA)
    record("course complaints", "PASS" if len(complaints) == 10000 else "FIX",
           f"{len(complaints):,} rows (expected 10,000)", FIX_DATA)
    # to_dict() turns the counts into a dictionary such as {"train": 6000, ...}.
    split_counts = complaints["split"].value_counts().to_dict()
    split_ok = split_counts == {"train": 6000, "valid": 2000, "test": 2000}

    # Write the counts as text, for example "train 6,000"; {:,} adds the thousands comma.
    split_parts = []
    for split_name, count in split_counts.items():
        split_parts.append(f"{split_name} {count:,}")
    record("course data split", "PASS" if split_ok else "FIX", ", ".join(split_parts), FIX_DATA)
else:
    record("read data files", "SKIP", SKIPPED, "Fix the FIX items above, then choose Restart and Run All.")

### 1.7 Cached embeddings

From Module 5 on, some labs use **embeddings**: lists of numbers that describe the meaning of a text. These labs load saved embeddings from `data/embeddings/`. You do not need to download or run a language model. Results saved like this for reuse are called a **cache**.

Each saved file has a **manifest**: a small JSON file that records its size and fingerprint. The cell lists the eight saved files the labs need and checks each one. Expect eight `PASS` lines. A missing or changed file gives `FIX`.

In [ ]:
# The eight saved embedding files that the labs use.
EMBEDDINGS = ["cfpb_chunks_minilm", "cfpb_minilm", "clothing_minilm", "feedback_minilm",
              "m05_sentences_minilm", "m06_queries_minilm", "m12_passages_minilm", "m12_questions_minilm"]
EMBEDDINGS_DIR = REPO / "data/embeddings"

for name in EMBEDDINGS:
    manifest = EMBEDDINGS_DIR / (name + "_manifest.json")
    if not ready:
        record("embeddings " + name + ".npy", "SKIP", SKIPPED)
        continue
    if not manifest.exists():
        record("embeddings " + name + ".npy", "FIX", "manifest not found", FIX_DATA)
        continue

    # Read the manifest into a dictionary: file name, rows, dimensions, fingerprint.
    info = json.loads(manifest.read_text())
    vectors_file = EMBEDDINGS_DIR / info["cache"]

    # Does the saved file exist and match its fingerprint?
    same = False
    if vectors_file.exists():
        same = hashlib.sha256(vectors_file.read_bytes()).hexdigest() == info["cache_sha256"]

    # np.load reads the array; its shape must be (rows, numbers per vector).
    ok = same and np.load(vectors_file).shape == (info["rows"], info["dimensions"])

    if same:
        hash_text = "matches"
    else:
        hash_text = "does NOT match"
    detail = f"{info['rows']:,} vectors of {info['dimensions']} numbers, sha256 {hash_text}"
    record("embeddings " + info["cache"], "PASS" if ok else "FIX", detail, FIX_DATA)

### 1.8 All checks so far

The table collects every check. Look at the `status` column. Rows marked `FIX` say what to do in `what_to_do`. Rows marked `SKIP` were not checked, because an earlier item needs a fix. Fix the `FIX` rows, then choose **Restart** and **Run All**.

In [ ]:
# One row per check, built from the dictionaries that record() kept.
pd.DataFrame(checks.values())

## Part 2 pandas warm-up

The labs assume you can read and summarize a table with pandas. These short steps refresh the moves you will use most. The **Python and pandas refresher** page in Canvas explains each one in more detail.

Part 2 uses the data that step 1.6 reads. If step 1.6 was skipped, fix the `FIX` rows first; the cells below cannot run without the data.

The first dataset is small: 24 short customer messages to an electronics shop. Every lab starts with it, so you can trace each number back to a message.

### 2.1 Look at the first rows

`head()` shows the first five rows. `shape` gives (rows, columns). Expect 24 rows and 5 columns: `feedback_id`, `channel`, `product`, `stars` and `text`.

In [ ]:
# shape is (rows, columns); head() shows the first five rows.
print(feedback.shape)
feedback.head()

### 2.2 Count values in a column

`value_counts()` counts how often each value appears. Expect 8 messages in each channel: review, email and chat.

In [ ]:
# Count the messages in each channel, most common first.
feedback["channel"].value_counts()

### 2.3 Missing text and text length

Real text data has gaps. `isna().sum()` counts missing values: one message (FB022) has no text. To count words, replace the missing text with an empty string, split each text on spaces and take the length of the list. `describe()` then summarizes the new column. Its rows are `count` (how many values), `mean` (average) and `std` (standard deviation, the typical distance from the mean). Then come `min`, the quartiles `25%`, `50%` and `75%`, and `max`. The `50%` row is the median: at least half the values are at or below it, and at least half are at or above it.

In [ ]:
print("messages with no text:", feedback["text"].isna().sum())
# fillna("") replaces the missing text with an empty string, so split() works on every row.
# .str applies a text method to every row: split() makes a list of words, len() counts them.
feedback["n_words"] = feedback["text"].fillna("").str.split().str.len()
feedback["n_words"].describe().round(1)

### 2.4 Filter rows

A condition in `.loc[...]` keeps only the rows where it is true. Here: messages with 1 or 2 stars, with four columns. Expect 8 rows.

In [ ]:
# .loc[rows to keep, columns to keep]
low_stars = feedback.loc[feedback["stars"] <= 2, ["feedback_id", "product", "stars", "text"]]
low_stars

### 2.5 Group and summarize

`groupby` splits the table into groups, and `agg` computes one summary per group. Each summary is written as `new_name=(column, function)`.

In [ ]:
# One row per product: number of messages, average stars, average words.
feedback.groupby("product").agg(
    messages=("feedback_id", "count"),
    mean_stars=("stars", "mean"),
    mean_words=("n_words", "mean"),
).round(2)

### 2.6 The course complaint data

The second dataset holds 10,000 real complaints that consumers sent to the U.S. Consumer Financial Protection Bureau (CFPB) in July–December 2025. It is the Practice data in most labs and one of the default project data options. Labs 4, 5 and 7 use public clothing reviews for Practice instead. `pd.crosstab` counts rows for each pair of values: here, product by split. `margins=True` adds the totals.

In [ ]:
print(complaints.columns.tolist())
# Rows: products. Columns: splits. margins=True adds an "All" row and column.
pd.crosstab(complaints["product"], complaints["split"], margins=True)

### 2.7 Try it yourself (optional)

These three tries use the same moves. They are not graded.

**Try 1.** In the 24 customer messages, how many messages in each channel have 1 or 2 stars? *Hint:* filter first, then use `value_counts()`.

In [ ]:
# Try 1: your code here
# TODO: keep the messages with stars <= 2, then use value_counts() on their channel.

**Try 2.** In the course data, which company has the most `Credit card` complaints, and how many? *Hint:* filter on `product`, then `value_counts()` on `company`.

In [ ]:
# Try 2: your code here
# TODO: keep the complaints whose product is "Credit card", then use value_counts() on company.

**Try 3.** In the course data, what is the median length of the `narrative` in words for each product? Sort from longest to shortest. *Hint:* step 2.3 shows how to count words; then `groupby` and `median()`.

In [ ]:
# Try 3: your code here
# TODO: count the words in each narrative (as in 2.3) and store them in a new column.
# TODO: group by product, take the median, and sort with sort_values(ascending=False).

## Part 3 Save and commit

### 3.1 Save the readiness report

Labs save their results to `outputs/`, never to `data/`. The cell creates `outputs/m00/` if needed and saves two files: the table of checks (`readiness_report.csv`) and a short summary with your versions (`readiness_summary.txt`). Neither file contains your user name.

In [ ]:
OUTPUT_DIR = Path("../outputs/m00")
# parents=True also creates outputs/ if needed; exist_ok=True means an existing folder is fine.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save the table of checks.
report = pd.DataFrame(checks.values())
report.to_csv(OUTPUT_DIR / "readiness_report.csv", index=False)

# Count PASS, FIX and NOTE; get(name, 0) gives 0 when a status does not occur.
status = report["status"].value_counts()

# The versions as text, for example "python 3.13.5, numpy 2.3.1".
version_parts = []
for name, number in versions.items():
    version_parts.append(name + " " + number)

# Four lines of summary text.
summary = [
    "BAN 6403 readiness check, " + datetime.now().strftime("%Y-%m-%d %H:%M"),
    "computer: " + platform.system() + " " + platform.release(),
    "versions: " + ", ".join(version_parts),
    f"checks: {len(report)}, PASS {status.get('PASS', 0)}, FIX {status.get('FIX', 0)}, NOTE {status.get('NOTE', 0)}",
]
# "\n".join puts each line on its own line in the file.
(OUTPUT_DIR / "readiness_summary.txt").write_text("\n".join(summary) + "\n")
print("\n".join(summary))

### 3.2 Read the report back

Reading a saved file back is the simplest proof that it was saved. The last lines give your result.

In [ ]:
saved = pd.read_csv(OUTPUT_DIR / "readiness_report.csv")

# The two files that step 3.1 saved.
print("saved:", OUTPUT_DIR / "readiness_report.csv")
print("saved:", OUTPUT_DIR / "readiness_summary.txt")
print("checks in the saved report:", len(saved))

# The rows that still need a fix; .empty is True when there are none.
to_fix = saved[saved["status"] == "FIX"]
if to_fix.empty:
    print("READY: every required check passed. You are set up for Week 1.")
else:
    print(f"NOT READY YET: {len(to_fix)} check(s) need a fix.")
    print(to_fix[["item", "what_to_do"]].to_string(index=False))

### 3.3 Commit your work in GitHub Desktop

A **commit** is a saved snapshot of your repository with a short message. Committing often means you can always go back to a version that worked. **Pushing** sends your commits to your copy on GitHub.

**Local (Anaconda and GitHub Desktop)**

1. Save this notebook (**Ctrl+S** on Windows, **Cmd+S** on macOS).
2. Open **GitHub Desktop**. The **Changes** tab lists the files you changed, including this notebook.
3. Check that nothing in `data/` is listed. If it is, right-click it and choose **Discard Changes**.
4. In the **Summary** box at the bottom left, type `Module 0 readiness check`. Click **Commit to main**.
5. Click **Push origin** to send the commit to your copy on GitHub.

**Codespaces**

1. Save this notebook (**Ctrl+S**, or **Cmd+S** on macOS).
2. Open **Source Control** in the left bar (the branch icon). It lists the files you changed.
3. Check that nothing in `data/` is listed. If it is, right-click it and choose **Discard Changes**.
4. Type `Module 0 readiness check` as the message and click **Commit**. If VS Code asks to stage your changes, choose **Yes**.
5. Click **Sync Changes** to push the commit to your copy on GitHub.

Pushing puts your readiness report in your repository on GitHub. If you have not done it yet, paste your repository link into the Module 0 item **Your repository link** in Canvas.

If you see `FIX` lines that you cannot solve, post the item and the *What to do* text in the course Q&A discussion.

## Further reading

- [pandas: 10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html)
- [VS Code: Jupyter notebooks](https://code.visualstudio.com/docs/datascience/jupyter-notebooks)
- [GitHub Docs: committing and reviewing changes in GitHub Desktop](https://docs.github.com/en/desktop/making-changes-in-a-branch/committing-and-reviewing-changes-to-your-project-in-github-desktop)

Next, Module 1: you look at customer feedback through a manager's decision and run your first lab.